# DataCentric-Env — GRPO Training Notebook

**End-to-end runnable by a judge.** Trains Qwen2.5-3B-Instruct as a data quality agent using GRPO via TRL + Unsloth.

**Runtime required:** GPU (T4 or better). In Colab: `Runtime → Change runtime type → T4 GPU`

## What this notebook does
1. Installs Unsloth, TRL, and dependencies
2. Loads Qwen2.5-3B-Instruct in 4-bit with LoRA adapters
3. Builds a prompt dataset from live environment observations
4. Defines a **reward function** that calls the live environment `/step` endpoint
5. Trains with `GRPOTrainer` — the reward function IS the environment verifier
6. Inspects sampled generations during training to catch reward hacking
7. Saves the model using the **Unsloth merge path** (not naive `save_pretrained`)

> **Grader note:** Format compliance is Grader 1, completely independent of accuracy.
> All reward values are clamped to strictly `(0.001, 0.999)` — never 0.0 or 1.0.

In [ ]:
# Cell 1: Install dependencies
# Unsloth install — use the correct pip source for Colab
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'unsloth', 'trl>=0.12.0', 'transformers>=4.45.0',
    'accelerate', 'peft', 'datasets', 'requests', 'bitsandbytes'
], check=True)
print('All dependencies installed.')

In [ ]:
# Cell 2: Imports and configuration
import json, re, requests, torch
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

# Live environment URL — deployed on HuggingFace Spaces
ENV_URL = 'https://aswini-kumar-datacentric-env.hf.space'

# Verify the environment is reachable before training
health = requests.get(f'{ENV_URL}/health', timeout=30)
assert health.status_code == 200, f'Environment not reachable: {health.status_code}'
print(f'Environment health: {health.json()}')

SYSTEM_PROMPT = (
    'You are a data quality agent. You receive dataset statistics and must choose '
    'which specialist tool to call to improve the dataset so a downstream classifier '
    'performs better.\n\n'
    'Always respond with valid JSON in this exact format:\n'
    '{"agent": "<tool_name>", "target": "<column_or_all>", "strategy": "<strategy_name>"}\n\n'
    'Available tools: cleaner, augmenter, balancer, relabeler, validator\n'
    'Cleaner strategies: median_impute, mean_impute, drop_rows\n'
    'Balancer strategies: undersample\n'
    'Relabeler: use when labels are noisy, costs 2 budget points.'
)

def build_prompt(obs: dict) -> str:
    return SYSTEM_PROMPT + '\n\nCurrent state:\n' + json.dumps(obs, indent=2) + '\n\nYour action:'

print('Config ready. ENV_URL:', ENV_URL)

In [ ]:
# Cell 3: Load Qwen2.5-3B-Instruct in 4-bit with LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,  # auto
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print(f'Model loaded. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Cell 4: Build prompt dataset
# Collect N initial observations by calling /reset.
# GRPOTrainer will generate multiple completions per prompt and score each via reward_fn.

N_PROMPTS = 60  # number of distinct starting states
prompts = []

difficulties = ['easy'] * 30 + ['medium'] * 20 + ['hard'] * 10  # curriculum mix

for i, diff in enumerate(difficulties):
    obs = requests.post(f'{ENV_URL}/reset', json={'difficulty': diff}, timeout=30).json()
    prompts.append({
        'prompt': build_prompt(obs),
        'difficulty': diff,
        'initial_obs': json.dumps(obs),
    })
    if i % 10 == 0:
        print(f'  Collected {i+1}/{N_PROMPTS} prompts (difficulty={diff})')

dataset = Dataset.from_list(prompts)
print(f'Dataset ready: {len(dataset)} prompts')
print('Sample prompt (truncated):', dataset[0]['prompt'][:300])

In [ ]:
# Cell 5: Reward function — this IS the environment verifier
#
# TRL GRPOTrainer calls reward_fn(completions, prompts, **kwargs) -> list[float]
# For each completion:
#   1. Parse the JSON action from the model output
#   2. Call /reset to get a fresh env state
#   3. Call /step with the parsed action
#   4. Return the environment reward (already in (0.001, 0.999))
#
# Grader 1 (format compliance) fires FIRST, independently of all other graders.
# Final reward is always clamped by the server before returning.

def parse_action(text: str) -> dict:
    """Extract JSON action from model output. Returns fallback on parse failure."""
    text = text.strip()
    # Try to extract JSON block
    match = re.search(r'\{[^}]+\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass
    # Fallback — intentionally malformed so Grader 1 fires 0.001
    return {'agent': '__invalid__'}


def reward_fn(completions, prompts=None, **kwargs):
    """
    Called by GRPOTrainer once per batch.
    Returns list[float] of rewards, one per completion.
    All values are strictly in (0.001, 0.999) — enforced by the server.
    """
    rewards = []
    for completion in completions:
        try:
            action = parse_action(completion)
            # Fresh env state for each reward evaluation
            requests.post(f'{ENV_URL}/reset', json={}, timeout=20)
            result = requests.post(f'{ENV_URL}/step', json=action, timeout=20).json()
            reward = float(result.get('reward', 0.001))
            # Safety clamp — should already be clamped by server
            reward = max(0.001, min(0.999, reward))
        except Exception as e:
            print(f'  [reward_fn error] {e}')
            reward = 0.001  # format/connection failure -> minimum
        rewards.append(reward)
    return rewards


# Quick sanity check before training
test_completions = [
    '{"agent": "cleaner", "target": "all", "strategy": "median_impute"}',
    'some random text that is not JSON',
    '{"agent": "balancer", "strategy": "undersample"}',
]
test_rewards = reward_fn(test_completions)
print('Reward function sanity check:')
for c, r in zip(test_completions, test_rewards):
    print(f'  action={c[:60]!r:65s} -> reward={r:.4f}')

assert all(0.0 < r < 1.0 for r in test_rewards), 'Reward out of valid range!'
print('All rewards in (0.0, 1.0): PASS')

In [ ]:
# Cell 6: Generation inspection — reward hacking detection
#
# Run this BEFORE and AFTER training to verify:
#   - Agent produces valid JSON actions
#   - Reward rises because accuracy rises, NOT because of exploits
#   - Agent varies its tool selection based on observation

def inspect_generations(n_episodes=5, label='baseline'):
    print(f'\n=== Generation inspection: {label} ===')
    tool_counts = {}
    valid_json_count = 0
    rewards = []

    for ep in range(n_episodes):
        obs = requests.post(f'{ENV_URL}/reset', json={}, timeout=30).json()
        prompt = build_prompt(obs)

        FastLanguageModel.for_inference(model)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=80, temperature=0.7, do_sample=True)
        FastLanguageModel.for_training(model)

        response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        action = parse_action(response)
        agent = action.get('agent', '__invalid__')

        # Step in env
        result = requests.post(f'{ENV_URL}/step', json=action, timeout=20).json()
        reward = result.get('reward', 0.0)
        acc_before = result.get('info', {}).get('prev_accuracy', '?')
        acc_after  = result.get('info', {}).get('new_accuracy', '?')

        is_valid = isinstance(action.get('agent'), str) and action['agent'] in [
            'cleaner', 'augmenter', 'balancer', 'relabeler', 'validator'
        ]
        if is_valid:
            valid_json_count += 1
        tool_counts[agent] = tool_counts.get(agent, 0) + 1
        rewards.append(reward)

        print(f'  Ep{ep+1}: agent={agent:12s} reward={reward:.4f}  '
              f'acc {acc_before}->{acc_after}  '
              f'raw_output={response[:80]!r}')

    print(f'\n  Valid JSON rate: {valid_json_count}/{n_episodes}')
    print(f'  Tool distribution: {tool_counts}')
    print(f'  Mean reward: {sum(rewards)/len(rewards):.4f}')
    print()

    # Reward hacking checks
    if len(tool_counts) == 1:
        print('  ⚠️  WARNING: Agent always picks same tool — possible reward hacking!')
    else:
        print('  ✅ Tool diversity OK — no same-tool spamming detected.')

    return {'valid_json_rate': valid_json_count/n_episodes, 'mean_reward': sum(rewards)/len(rewards), 'tool_counts': tool_counts}


# Run BEFORE training — establishes the baseline
before_stats = inspect_generations(n_episodes=5, label='BEFORE training')

In [ ]:
# Cell 7: GRPO Training
#
# GRPOTrainer:
#   - samples num_generations completions per prompt
#   - calls reward_fn on all completions
#   - updates model to increase probability of higher-reward completions
#
# The reward_fn IS the environment verifier — no learned reward model needed.

config = GRPOConfig(
    output_dir='./datacentric-grpo',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,          # 4 completions per prompt, ranked by reward
    max_completion_length=100,  # action JSON is short
    learning_rate=5e-6,
    logging_steps=5,
    save_steps=50,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    report_to='none',           # swap to 'wandb' for live curves
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    reward_funcs=reward_fn,     # environment verifier IS the reward function
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print('Starting GRPO training...')
print(f'  Dataset: {len(dataset)} prompts')
print(f'  Generations per prompt: {config.num_generations}')
print(f'  Total reward evaluations per epoch: {len(dataset) * config.num_generations}')
print()

trainer.train()

print('Training complete.')

In [ ]:
# Cell 8: Post-training generation inspection
# Compare BEFORE vs AFTER to verify no reward hacking occurred.
# Expected: higher valid JSON rate, higher mean reward, varied tool selection.

after_stats = inspect_generations(n_episodes=5, label='AFTER training')

print('=== Before vs After comparison ===')
print(f'  Valid JSON rate: {before_stats["valid_json_rate"]:.1%} -> {after_stats["valid_json_rate"]:.1%}')
print(f'  Mean reward:     {before_stats["mean_reward"]:.4f} -> {after_stats["mean_reward"]:.4f}')
print(f'  Tool diversity:  {len(before_stats["tool_counts"])} tools -> {len(after_stats["tool_counts"])} tools')

if after_stats['mean_reward'] > before_stats['mean_reward']:
    print('  ✅ Reward improved after training — learning signal working.')
else:
    print('  ⚠️  Reward did not improve — check env connectivity and reward function.')

In [ ]:
# Cell 9: Save model using Unsloth merge path
#
# CRITICAL: Do NOT use naive save_pretrained on a 4-bit model.
# That upcasts to 16-bit then merges naively, which damages model quality.
# Use save_pretrained_merged with save_method='merged_16bit' instead.

SAVE_DIR = 'datacentric-grpo-final'

model.save_pretrained_merged(
    SAVE_DIR,
    tokenizer,
    save_method='merged_16bit',   # correct Unsloth merge path
)

print(f'Model saved to {SAVE_DIR}/')
print('Test inference immediately to verify model quality before uploading.')

# Quick inference test
from transformers import AutoModelForCausalLM, AutoTokenizer
test_tok = AutoTokenizer.from_pretrained(SAVE_DIR)
test_model = AutoModelForCausalLM.from_pretrained(SAVE_DIR, torch_dtype=torch.float16, device_map='auto')

obs = requests.post(f'{ENV_URL}/reset', json={}, timeout=30).json()
inp = test_tok(build_prompt(obs), return_tensors='pt').to('cuda')
with torch.no_grad():
    out = test_model.generate(**inp, max_new_tokens=80)
response = test_tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'Post-save inference test: {response}')
action = parse_action(response)
print(f'Parsed action: {action}')
print('Model save and inference verified.')